In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2 as cv
import random
import warnings
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import os
from tqdm import tqdm
import albumentations as A
from torchvision.models import resnet50, ResNet50_Weights



In [2]:
SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n Device : {device.upper()}")





 Device : CUDA


In [3]:
def resolve_path(path: str) -> str:
    if path is None:
        return path
    if os.path.exists(path):
        return path

    p = path.replace("\\", "/")

    for anchor in ("/input/", "/data/"):
        if anchor in p:
            suffix = p.split(anchor, 1)[1]
            for base in ("/kaggle/input/", "/kaggle/data/"):
                cand = os.path.join(base, suffix)
                if os.path.exists(cand):
                    return cand

    for prefix in ("../input/", "../data/"):
        if p.startswith(prefix):
            suffix = p[len(prefix) :]
            for base in ("/kaggle/input/", "/kaggle/data/"):
                cand = os.path.join(base, suffix)
                if os.path.exists(cand):
                    return cand

    return path  # fall back; may error later with a clear message


# Change (score-relevant reliability): robustly find the actual image directories.
# Kaggle datasets sometimes appear in multiple roots; if we point to a non-existent folder,
# cv2.imread returns None -> crash/no submission. This keeps core logic identical.
def resolve_image_dir(preferred_path: str, expected_subdir_name: str) -> str:
    candidates = []
    if preferred_path:
        candidates.append(preferred_path)

    # Try same parent as CSVs if present
    for base in (
        "/kaggle/input/aptos2019-blindness-detection",
        "/kaggle/data/aptos2019-blindness-detection",
    ):
        candidates.append(os.path.join(base, expected_subdir_name))

    # Try flat roots (as shown in your file listing)
    for base in ("/kaggle/input", "/kaggle/data"):
        candidates.append(os.path.join(base, expected_subdir_name))

    # Search for first matching directory that contains at least one .png
    for c in candidates:
        c = resolve_path(c)
        if os.path.isdir(c):
            try:
                for fn in os.listdir(c):
                    if fn.lower().endswith(".png"):
                        return c
            except Exception:
                pass

    # Slow fallback: walk a bit but stop early (still fast enough)
    for root in ("/kaggle/input", "/kaggle/data"):
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            if os.path.basename(dirpath) != expected_subdir_name:
                continue
            for fn in filenames:
                if fn.lower().endswith(".png"):
                    return dirpath

    return preferred_path


TEST_PATH = resolve_path("../input/aptos2019-blindness-detection/test.csv")
SAMPLE_SUB_PATH = resolve_path(
    "../input/aptos2019-blindness-detection/sample_submission.csv"
)
TRAIN_PATH = resolve_path("../input/aptos2019-blindness-detection/train.csv")

TEST_IMG = resolve_image_dir(
    resolve_path("../input/aptos2019-blindness-detection/test_images"), "test_images"
)
TRAIN_IMG = resolve_image_dir(
    resolve_path("../input/aptos2019-blindness-detection/train_images"), "train_images"
)

print("Resolved paths:")
print(" TEST_PATH:", TEST_PATH)
print(" SAMPLE   :", SAMPLE_SUB_PATH)
print(" TRAIN    :", TRAIN_PATH)
print(" TEST_IMG :", TEST_IMG)
print(" TRAIN_IMG:", TRAIN_IMG)

WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
os.makedirs(WORKDIR, exist_ok=True)




Resolved paths:
 TEST_PATH: ../input/aptos2019-blindness-detection/test.csv
 SAMPLE   : ../input/aptos2019-blindness-detection/sample_submission.csv
 TRAIN    : ../input/aptos2019-blindness-detection/train.csv
 TEST_IMG : ../input/aptos2019-blindness-detection/test_images
 TRAIN_IMG: ../input/aptos2019-blindness-detection/train_images


In [4]:
class AptosDataset(Dataset):
    def __init__(
        self,
        data_path,
        img_dir,
        name,
        transforms,
        resize=(512, 512),
        return_label=False,
    ):
        self.data_path = data_path
        self.img_dir = img_dir
        self.resize = resize
        self.transforms = transforms
        self.df = pd.read_csv(self.data_path)
        self.name = name
        self.return_label = return_label

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row["id_code"]
        img_name = img_id + ".png"
        img_path = os.path.join(self.img_dir, img_name)

        img = cv.imread(img_path, cv.IMREAD_COLOR)  # BGR uint8
        if img is None:
            raise FileNotFoundError(f"Could not read image: {img_path}")

        img = cv.cvtColor(img, cv.COLOR_BGR2RGB)

        if self.transforms:
            transformed = self.transforms(image=img)
            img = transformed["image"]
        else:
            if self.resize:
                img = cv.resize(img, self.resize, interpolation=cv.INTER_AREA)
            img = img.astype(np.float32) / 255.0

        if img.dtype != np.float32:
            img = img.astype(np.float32)

        img = np.ascontiguousarray(img.transpose(2, 0, 1))
        img_t = torch.from_numpy(img).float()

        if self.return_label:
            y = int(row["diagnosis"])
            return img_t, y
        return img_t




In [5]:
def find_first_existing(paths):
    for p in paths:
        if p and os.path.isfile(p):
            return p
    return None


def _unwrap_checkpoint_to_state_dict(obj):
    if not isinstance(obj, dict):
        return None
    if len(obj) > 0 and any(isinstance(k, str) and "." in k for k in obj.keys()):
        return obj
    for key in ("state_dict", "model_state_dict", "model", "net", "weights"):
        v = obj.get(key, None)
        if isinstance(v, dict) and len(v) > 0:
            return v
    return obj


def _strip_prefix(state, prefixes=("module.", "model.", "net.")):
    if not isinstance(state, dict) or len(state) == 0:
        return state
    out = {}
    for k, v in state.items():
        nk = k
        for p in prefixes:
            if nk.startswith(p):
                nk = nk[len(p) :]
        out[nk] = v
    return out


candidate_weight_paths = [
    "../input/pretrained/model-2.bin",
    "/kaggle/input/pretrained/model-2.bin",
    "../input/pretrained/model-2.pth",
    "/kaggle/input/pretrained/model-2.pth",
    "../input/pretrained/model-2.pt",
    "/kaggle/input/pretrained/model-2.pt",
]

if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        for fn in files:
            if fn in ("model-2.bin", "model-2.pth", "model-2.pt"):
                candidate_weight_paths.append(os.path.join(root, fn))

WEIGHTS_PATH = find_first_existing(candidate_weight_paths)
print(
    "Competition weights found at:",
    (
        WEIGHTS_PATH
        if WEIGHTS_PATH
        else "NOT FOUND (will fall back to ImageNet pretrained backbone)"
    ),
)



Competition weights found at: NOT FOUND (will fall back to ImageNet pretrained backbone)


In [6]:
BATCH_SIZE = 16
IMG_DIM = 512

NUM_WORKERS = 2 if os.cpu_count() and os.cpu_count() >= 4 else 0

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

test_tfms = A.Compose(
    [
        A.SmallestMaxSize(max_size=IMG_DIM, interpolation=cv.INTER_AREA),
        A.CenterCrop(height=IMG_DIM, width=IMG_DIM),
        A.ToFloat(max_value=255.0),
        A.Normalize(mean=imagenet_mean, std=imagenet_std, max_pixel_value=1.0),
    ]
)

train_tfms = test_tfms


# Change (score-relevant reliability): quick preflight to ensure images can be read from resolved dirs.
# Avoids "Not yielded" due to late crash; does not change any modeling logic.
def preflight_check_images(csv_path, img_dir, n=3):
    df = pd.read_csv(csv_path)
    n = min(n, len(df))
    for i in range(n):
        img_id = df.iloc[i]["id_code"]
        p = os.path.join(img_dir, img_id + ".png")
        img = cv.imread(p, cv.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(f"Preflight failed: cannot read {p}")
    print(f"Preflight OK for {n} images in {img_dir}")


preflight_check_images(TEST_PATH, TEST_IMG, n=3)
preflight_check_images(TRAIN_PATH, TRAIN_IMG, n=3)

test_ids_csv = os.path.join(WORKDIR, "test_ids_from_sample.csv")
pd.read_csv(SAMPLE_SUB_PATH)[["id_code"]].to_csv(test_ids_csv, index=False)

test_dataset = AptosDataset(
    test_ids_csv,
    TEST_IMG,
    "test",
    transforms=test_tfms,
    resize=(IMG_DIM, IMG_DIM),
    return_label=False,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
)

if WEIGHTS_PATH is None:
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
else:
    model = resnet50(weights=None)

model.fc = nn.Sequential(
    nn.Linear(in_features=2048, out_features=1024, bias=True),
    nn.Linear(in_features=1024, out_features=512, bias=True),
    nn.Linear(in_features=512, out_features=5, bias=True),
)

if WEIGHTS_PATH is not None:
    raw = torch.load(WEIGHTS_PATH, map_location="cpu")
    state = _unwrap_checkpoint_to_state_dict(raw)
    state = _strip_prefix(state)

    try:
        model.load_state_dict(state, strict=True)
        print("Loaded competition weights with strict=True")
    except RuntimeError as e:
        print(
            "Strict weight load failed; retrying with strict=False. Error was:\n",
            str(e)[:800],
        )
        incompat = model.load_state_dict(state, strict=False)
        missing = list(getattr(incompat, "missing_keys", []))
        unexpected = list(getattr(incompat, "unexpected_keys", []))
        print(
            "Loaded with strict=False. Missing keys:",
            len(missing),
            "Unexpected keys:",
            len(unexpected),
        )

model = model.to(device)
model.eval()




Preflight OK for 3 images in ../input/aptos2019-blindness-detection/test_images
Preflight OK for 3 images in ../input/aptos2019-blindness-detection/train_images


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 109MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [7]:
def quadratic_weighted_kappa(y_true, y_pred, n_classes=5):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    assert y_true.shape == y_pred.shape

    O = np.zeros((n_classes, n_classes), dtype=np.float64)
    for a, b in zip(y_true, y_pred):
        if 0 <= a < n_classes and 0 <= b < n_classes:
            O[a, b] += 1.0

    act_hist = O.sum(axis=1)
    pred_hist = O.sum(axis=0)
    E = np.outer(act_hist, pred_hist)
    if E.sum() > 0:
        E = E * (O.sum() / E.sum())

    W = np.zeros((n_classes, n_classes), dtype=np.float64)
    for i in range(n_classes):
        for j in range(n_classes):
            W[i, j] = ((i - j) ** 2) / ((n_classes - 1) ** 2)

    num = (W * O).sum()
    den = (W * E).sum()
    return 1.0 - (num / den if den > 0 else 0.0)


def logits_to_continuous_score(logits_5):
    logits_5 = np.asarray(logits_5, dtype=np.float32)
    p = (
        torch.softmax(torch.from_numpy(logits_5), dim=1)
        .cpu()
        .numpy()
        .astype(np.float32)
    )
    cdf = np.cumsum(p, axis=1)  # P(y <= k)
    score = 4.0 - cdf[:, :4].sum(axis=1)
    return score.astype(np.float32)


def apply_thresholds(scores, thresholds):
    scores = np.asarray(scores, dtype=np.float32)
    t = np.asarray(thresholds, dtype=np.float32)
    pred = np.zeros(scores.shape[0], dtype=int)
    pred[scores > t[0]] = 1
    pred[scores > t[1]] = 2
    pred[scores > t[2]] = 3
    pred[scores > t[3]] = 4
    return pred


def fit_thresholds_by_grid(
    scores, y_true, base_thresholds=(0.5, 1.5, 2.5, 3.5), step=0.05, span=0.8
):
    scores = np.asarray(scores, dtype=np.float32)
    y_true = np.asarray(y_true, dtype=int)

    base = np.asarray(base_thresholds, dtype=np.float32)
    grids = []
    for b in base:
        lo = b - span
        hi = b + span
        grids.append(np.arange(lo, hi + 1e-9, step, dtype=np.float32))

    best_kappa = -1e9
    best_t = base.copy()

    for t0 in grids[0]:
        for t1 in grids[1]:
            if t1 <= t0:
                continue
            for t2 in grids[2]:
                if t2 <= t1:
                    continue
                for t3 in grids[3]:
                    if t3 <= t2:
                        continue
                    pred = apply_thresholds(scores, (t0, t1, t2, t3))
                    k = quadratic_weighted_kappa(y_true, pred, n_classes=5)
                    if k > best_kappa:
                        best_kappa = k
                        best_t = np.array([t0, t1, t2, t3], dtype=np.float32)
    return best_t, float(best_kappa)


def fit_thresholds_coarse_to_fine(
    scores,
    y_true,
    base_thresholds=(0.5, 1.5, 2.5, 3.5),
    span=0.6,
    coarse_step=0.20,
    fine_step=0.05,
    fine_span=0.20,
):
    t_coarse, k_coarse = fit_thresholds_by_grid(
        scores,
        y_true,
        base_thresholds=base_thresholds,
        step=coarse_step,
        span=span,
    )
    t_fine, k_fine = fit_thresholds_by_grid(
        scores,
        y_true,
        base_thresholds=tuple(float(x) for x in t_coarse.tolist()),
        step=fine_step,
        span=fine_span,
    )
    if k_fine >= k_coarse:
        return t_fine, k_fine
    return t_coarse, k_coarse




In [8]:
def make_stratified_subset(df, max_n=1200, seed=SEED):
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    max_n = min(max_n, len(df))
    per_class = max(1, max_n // 5)
    parts = []
    for c in range(5):
        df_c = df[df["diagnosis"] == c]
        take = min(per_class, len(df_c))
        if take > 0:
            parts.append(df_c.iloc[:take])
    out = pd.concat(parts, axis=0).reset_index(drop=True)
    if len(out) < max_n:
        remaining = df[~df["id_code"].isin(out["id_code"])].reset_index(drop=True)
        need = max_n - len(out)
        out = pd.concat([out, remaining.iloc[:need]], axis=0).reset_index(drop=True)
    return out


def warmup_head(model, train_df, epochs=2, lr=3e-4):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.fc.parameters():
        p.requires_grad = True

    model.train()

    warm_csv = os.path.join(WORKDIR, "warmup_subset.csv")
    train_df.to_csv(warm_csv, index=False)

    ds = AptosDataset(
        warm_csv,
        TRAIN_IMG,
        "warmup",
        transforms=train_tfms,
        resize=(IMG_DIM, IMG_DIM),
        return_label=True,
    )
    dl = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(NUM_WORKERS > 0),
    )

    opt = torch.optim.AdamW(model.fc.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()

    for ep in range(epochs):
        total = 0.0
        n = 0
        for x, y in tqdm(dl, desc=f"Warmup head epoch {ep+1}/{epochs}", leave=False):
            x = x.to(device)
            y = y.to(device)

            opt.zero_grad(set_to_none=True)
            out = model(x)
            loss = crit(out, y)
            loss.backward()
            opt.step()

            bs = y.size(0)
            total += float(loss.detach().cpu()) * bs
            n += bs
        print(f"Warmup epoch {ep+1}: loss={total/max(1,n):.4f}")

    model.eval()


train_df_full = pd.read_csv(TRAIN_PATH)

if WEIGHTS_PATH is None:
    warm_df = make_stratified_subset(train_df_full, max_n=1000, seed=SEED)
    warmup_head(model, warm_df, epochs=2, lr=3e-4)



Warmup epoch 1: loss=1.2670


Warmup epoch 2: loss=0.9791


In [9]:
train_df = train_df_full.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

max_calib = min(1200, len(train_df))
per_class = max(1, max_calib // 5)
parts = []
for c in range(5):
    df_c = train_df[train_df["diagnosis"] == c]
    take = min(per_class, len(df_c))
    if take > 0:
        parts.append(df_c.iloc[:take])
calib_df = pd.concat(parts, axis=0).reset_index(drop=True)

if len(calib_df) < max_calib:
    remaining = train_df[~train_df["id_code"].isin(calib_df["id_code"])].reset_index(
        drop=True
    )
    need = max_calib - len(calib_df)
    calib_df = pd.concat([calib_df, remaining.iloc[:need]], axis=0).reset_index(
        drop=True
    )

fold = np.full(len(calib_df), -1, dtype=int)
for c in range(5):
    idx_c = np.where(calib_df["diagnosis"].values.astype(int) == c)[0]
    fold[idx_c] = (np.arange(len(idx_c)) % 2).astype(int)


def infer_logits_for_df(df_subset, csv_name):
    csv_path = os.path.join(WORKDIR, csv_name)
    df_subset.to_csv(csv_path, index=False)

    ds = AptosDataset(
        csv_path,
        TRAIN_IMG,
        "calib_part",
        transforms=train_tfms,
        resize=(IMG_DIM, IMG_DIM),
        return_label=True,
    )
    dl = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(NUM_WORKERS > 0),
    )
    logits_list = []
    y_list = []
    with torch.no_grad():
        for x, y in tqdm(dl, desc=f"Infer {csv_name}", leave=False):
            out = model(x.to(device))
            logits_list.append(out.detach().cpu())
            y_list.extend(y.numpy().tolist())
    logits = torch.cat(logits_list, dim=0).numpy()
    return logits, np.asarray(y_list, dtype=int)


oof_scores = np.zeros(len(calib_df), dtype=np.float32)
oof_y = calib_df["diagnosis"].astype(int).values

for f in (0, 1):
    va_idx = np.where(fold == f)[0]
    va_df = calib_df.iloc[va_idx].reset_index(drop=True)
    va_logits, va_y = infer_logits_for_df(va_df, f"calib_fold{f}.csv")
    va_scores = logits_to_continuous_score(va_logits)

    if len(va_scores) != len(va_idx) or len(va_y) != len(va_idx):
        raise RuntimeError("OOF inference size mismatch.")
    oof_scores[va_idx] = va_scores.astype(np.float32)

thresholds, best_k = fit_thresholds_coarse_to_fine(
    oof_scores,
    oof_y,
    base_thresholds=(0.5, 1.5, 2.5, 3.5),
    span=0.6,
    coarse_step=0.20,
    fine_step=0.05,
    fine_span=0.20,
)
print("Fitted thresholds (OOF):", thresholds.tolist(), "OOF QWK:", best_k)



Fitted thresholds (OOF): [1.1000001430511475, 2.1000006198883057, 2.5999996662139893, 3.049999713897705] OOF QWK: 0.8455575432319619


In [10]:
test_logits = []
with torch.no_grad():
    for x in tqdm(test_loader, desc="Inference", leave=False):
        logits = model(x.to(device))
        test_logits.append(logits.detach().cpu())
test_logits = torch.cat(test_logits, dim=0).numpy()

test_scores = logits_to_continuous_score(test_logits)
labels = apply_thresholds(test_scores, thresholds).tolist()

print("Preds:", len(labels), "Expected:", len(test_dataset))

sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
if len(labels) != len(sample_sub):
    raise RuntimeError(
        f"Prediction length mismatch: got {len(labels)} preds, submission expects {len(sample_sub)} rows"
    )

sample_sub["diagnosis"] = labels
out_path = os.path.join(WORKDIR, "submission.csv")
sample_sub.to_csv(out_path, index=False)
print("Wrote submission.csv with shape:", sample_sub.shape)
print("submission.csv path:", os.path.abspath(out_path))
print(sample_sub.head())

Preds: 367 Expected: 367
Wrote submission.csv with shape: (367, 2)
submission.csv path: /kaggle/working/submission.csv
        id_code  diagnosis
0  b460ca9fa26f          0
1  6cee2e148520          0
2  ca6842bfcbc9          1
3  6cbc3dad809c          3
4  a9bc2f892cb3          0
